# ContractRisk — verificación de despliegue 17

Corrige la instalación fallida de la prueba 16: Gradio 6.32.0 y huggingface-hub 0.36.2 son incompatibles. Esta versión fija **Gradio 6.16.0**, conserva las bibliotecas/modelos de inferencia evaluados y comprueba las dependencias antes de cargar pesos. No instala con --no-deps ni ignora el fallo anterior. La evidencia 16 permanece intacta.

Usar una **sesión nueva de Colab con GPU T4**. Ejecutar en orden; si falla instalación o comprobación, no ejecutar la inferencia: pasar a la última celda para conservar el diagnóstico. No requiere claves, Gemini ni pagos. No cambia main ni el Space.

La prueba comprueba M1, texto con fuentes, cálculo, captura sintética y abstenciones. No repite el benchmark semántico ni demuestra el ciclo de vida de ZeroGPU. El resultado requiere revisión antes de publicar.

In [ ]:
import os,sys,subprocess,tempfile,json
from pathlib import Path
os.environ['USE_TF']='0'
os.environ['TOKENIZERS_PARALLELISM']='false'
os.environ['ANONYMIZED_TELEMETRY']='false'
SETUP_OK=False
ROOT=Path(tempfile.mkdtemp(prefix='contractrisk_deployment17_',dir='/content'))/'repo'
subprocess.run(['git','clone','--single-branch','--branch','Entrega/contractrisk-final','https://github.com/vcastrop/Juan-Camilo-Ramon-Valentina-Castro.git',str(ROOT)],check=True)
os.chdir(ROOT)
COMMIT=subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()
print('Código:',COMMIT)
def run_visible(arguments,logfile):
    with open(logfile,'w',encoding='utf-8') as log:
        process=subprocess.Popen(arguments,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            print(line,end='',flush=True);log.write(line);log.flush()
        status=process.wait()
    if status:raise RuntimeError('Proceso detenido. Conserva el diagnóstico con la última celda: '+logfile)


In [ ]:
SETUP_OK=False
run_visible([sys.executable,'final/prepare_test_history.py'],'deployment17_history.log')
run_visible([sys.executable,'-m','pip','install','-r','final/space_candidate/requirements.txt'],'deployment17_install.log')
run_visible([sys.executable,'final/service_environment.py','--output','deployment17_environment.json'],'deployment17_environment.log')
run_visible(['apt-get','update'],'deployment17_apt.log')
run_visible(['apt-get','install','-y','fonts-dejavu-core'],'deployment17_fonts.log')
run_visible([sys.executable,'-c',"import torch; assert torch.cuda.is_available() and 'T4' in torch.cuda.get_device_name(0), 'Selecciona GPU T4'; print(torch.cuda.get_device_name(0))"],'deployment17_gpu.log')
run_visible([sys.executable,'-m','unittest','discover','-s','final/tests','-v'],'deployment17_tests_final.log')
run_visible([sys.executable,'-m','unittest','discover','-s','m3/tests','-v'],'deployment17_tests_m3.log')
SETUP_OK=True
print('Preparación comprobada. Se puede ejecutar la prueba de modelos reales.')


In [ ]:
if not SETUP_OK:raise RuntimeError('La preparación no terminó correctamente. No ejecutar inferencia; descargar el diagnóstico con la última celda.')
CANDIDATE=ROOT/'deployment17_candidate'
run_visible([sys.executable,'final/build_space_candidate.py','--output',str(CANDIDATE)],'deployment17_build.log')
RESULTS=ROOT/'deployment17_results'
run_visible([sys.executable,'-u','final/run_service_preflight.py','--candidate',str(CANDIDATE),'--output',str(RESULTS)],'deployment17_preflight.log')


## Evidencia e interpretación

`completed_requires_review` indica finalización técnica, no aprobación de las respuestas. Revisar fuentes, cálculos, clasificación y abstenciones en los archivos. Si falla, conservar el parcial con la siguiente celda; no sustituir resultados ni cambiar paquetes manualmente.

El ZIP y este notebook descargado con sus salidas permiten auditar la ejecución. No subir todavía el paquete al Space. Los resultados guardados aquí no son los registros públicos de W&B.

In [ ]:
import zipfile
from google.colab import files
metadata=ROOT/'deployment17_results/metadata.json'
print(metadata.read_text(encoding='utf-8') if metadata.exists() else 'La prueba no terminó o no comenzó. Consultar los registros conservados.')
archive=ROOT.parent/'contractrisk_final_despliegue17_resultados.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for folder in ['deployment17_candidate','deployment17_results']:
        base=ROOT/folder
        if base.exists():
            for item in sorted(base.rglob('*')):
                if item.is_file():z.write(item,item.relative_to(ROOT))
    for pattern in ['deployment17_*.log','deployment17_environment.json']:
        for item in sorted(ROOT.glob(pattern)):z.write(item,item.name)
    z.writestr('checkout_commit.txt',COMMIT+'\n')
    z.writestr('setup_state.json',json.dumps({'setup_ok':SETUP_OK,'inference_allowed_only_after_setup':True}))
files.download(str(archive))
